# 01 - Data Preparation

Cleans and chunks 5 Turkish statutes from `muhammetakkurt/mevzuat-gov-dataset` (HuggingFace).

**Output:** `mevzuat_temiz.json` — one chunk per article, with next-section title appended

**Laws covered:**
- TCK (No. 5237) — Türk Ceza Kanunu
- TMK (No. 4721) — Türk Medeni Kanunu  
- İş K. (No. 4857) — İş Kanunu
- TBK (No. 6098) — Türk Borçlar Kanunu
- Anayasa (No. 2709) — Türkiye Cumhuriyeti Anayasası

## Installation

In [ ]:
!pip install datasets pandas transformers numpy -q

## Load Dataset

In [ ]:
from datasets import load_dataset
import pandas as pd

print("Loading dataset...")
mevzuat = load_dataset("muhammetakkurt/mevzuat-gov-dataset")
df = pd.DataFrame(mevzuat['train'])

print(f"Total laws: {len(df)}")
print(f"Columns: {df.columns.tolist()}

## Extract 5 Target Laws

In [ ]:
# Filter the 5 target statutes
KANUNLAR = {
    "5237": "Türk Ceza Kanunu",
    "4721": "Türk Medeni Kanunu",
    "4857": "İş Kanunu",
    "6098": "Türk Borçlar Kanunu",
    "2709": "Türkiye Cumhuriyeti Anayasası"
}

ham_kanunlar = {}
for numara, isim in KANUNLAR.items():
    satir = df[df['kanun_numarasi'] == numara].iloc[0]
    ham_kanunlar[numara] = {
        "isim": isim,
        "maddeler": satir['maddeler']
    }
    print(f"✅ {isim}: {len(satir['maddeler'])} articles")

## Helper Functions

Two shared utilities used by all statute-specific cleaners:

- `son_baslik_isle`: strips Kısım/Bölüm headers from section titles, keeping only the semantic title
- `madde_isle`: applies per-article cleaning (footnote removal, mülga filtering, title extraction)

In [ ]:
import re

def son_baslik_isle(baslik_ham):
    """
    Clean section title extracted from end of article text.
    Removes Kısım/Bölüm numbering blocks (e.g. 'İKİNCİ KISIM BİRİNCİ BÖLÜM')
    and keeps only the semantic title (e.g. 'Hırsızlık').
    """
    if not baslik_ham:
        return ""

    # Split on ordinal + KISIM/BÖLÜM pattern
    parcalar = re.split(
        r'(?:BİRİNCİ|İKİNCİ|ÜÇÜNCÜ|DÖRDÜNCÜ|BEŞİNCİ|ALTINCI|YEDİNCİ|'
        r'SEKİZİNCİ|DOKUZUNCU|ONUNCU|ON\s+\w+)\s+(?:KISIM|BÖLÜM)\s+',
        baslik_ham
    )
    son = parcalar[-1].strip()

    # Extract last title-cased phrase (starts with capital, continues lowercase)
    eslesme = re.findall(r'[A-ZÇĞİÖŞÜ][a-zçğışöü][^A-Z]*', son)
    if eslesme:
        return eslesme[-1].strip()
    return son


def madde_isle(metin):
    """
    Clean a single article's raw text:
    1. Remove footnote references ([1], [2], ...) and orphaned page numbers
    2. Detect fully-repealed (mülga) articles → return None
    3. Remove mülga paragraphs (fıkralar), rescuing any trailing section title
    4. Split off trailing next-section title from article body
    
    Returns: (icerik, sonraki_baslik) or (None, None) if fully repealed
    """
    # Step 1: Clean footnotes and orphaned page numbers
    metin = re.sub(r'\[\d+\]', '', metin)
    metin = re.sub(r'\.\s+\d+\s+', '. ', metin)

    # Step 2: Fully mülga?
    metin_temiz = metin.strip().lstrip('–').lstrip('-').strip()
    if metin_temiz.lower().startswith('(mülga'):
        return None, None

    # Step 3: Remove mülga paragraphs
    kurtarilan_baslik = ""
    if '(1)' in metin:
        # Split into numbered paragraphs (fıkralar)
        fikralar = re.split(r'(?=\(\d+\))', metin)
        fikralar = [f.strip() for f in fikralar if f.strip()]
        temiz = []
        for f in fikralar:
            if 'mülga' in f.lower():
                # Try to rescue a section title from the mülga paragraph
                eslesme = re.search(r'\)\s+([A-ZÇĞİÖŞÜ][^()]+)$', f)
                if eslesme:
                    kurtarilan_baslik = eslesme.group(1).strip()
            else:
                temiz.append(f)
        metin = ' '.join(temiz)
    else:
        # No numbered paragraphs — just strip mülga annotations
        metin = re.sub(r'\(Mülga[^)]*\)', '', metin, flags=re.IGNORECASE).strip()

    # Step 4: Split trailing section title
    son_nokta = metin.rfind('.')
    if son_nokta == -1:
        return metin.strip(), kurtarilan_baslik

    icerik = metin[:son_nokta + 1].strip()
    baslik_ham = metin[son_nokta + 1:].strip()

    # Clean the trailing title (strip Kısım/Bölüm numbering)
    baslik = son_baslik_isle(baslik_ham)

    # Fall back to rescued title if no trailing title found
    if not baslik and kurtarilan_baslik:
        baslik = kurtarilan_baslik

    return icerik, baslik

print("✅ Helper functions ready")

## TCK — Türk Ceza Kanunu (5237)

**Key challenges:**
- Duplicate articles from amendments (deduplicated by first-seen)
- Orphaned footnote numbers in text
- Next-section titles embedded at end of article text

In [ ]:
def tck_temizle(maddeler):
    """
    Clean TCK articles.
    - Deduplicates by raw madde_numarasi (preserving 'Geçici Madde X' format)
    - Skips fully-mülga articles, rescuing any next-section title
    - Delegates per-article cleaning to madde_isle()
    - Propagates bekleyen_baslik: when a mülga article holds a section title,
      it is attached to the preceding valid article
    """
    goruldu = set()
    sonuclar = []
    bekleyen_baslik = ""

    for madde in maddeler:
        metin = madde['text']
        madde_no = madde['madde_numarasi'].strip()  # preserve original case

        # Deduplicate
        if madde_no in goruldu:
            continue
        goruldu.add(madde_no)

        # Handle fully-mülga articles
        metin_temiz = metin.strip().lstrip('–').lstrip('-').strip()
        if metin_temiz.lower().startswith('(mülga'):
            son_parantez = metin.rfind(')')
            if son_parantez != -1:
                baslik = metin[son_parantez + 1:].strip()
                if baslik:
                    bekleyen_baslik = baslik
            continue

        icerik, sonraki_baslik = madde_isle(metin)
        if icerik is None:
            continue

        # Propagate bekleyen_baslik to previous article
        if bekleyen_baslik and sonuclar:
            sonuclar[-1]['sonraki_baslik'] = bekleyen_baslik
            sonuclar[-1]['encode_text'] = (
                sonuclar[-1]['icerik'] + f"\n\n{bekleyen_baslik}"
            )
            bekleyen_baslik = ""

        encode_text = icerik + (f"\n\n{sonraki_baslik}" if sonraki_baslik else "")

        sonuclar.append({
            "kanun": "Türk Ceza Kanunu",
            "madde_no": madde_no,
            "icerik": icerik,
            "sonraki_baslik": sonraki_baslik,
            "encode_text": encode_text
        })

    return sonuclar

tck_temiz = tck_temizle(ham_kanunlar["5237"]["maddeler"])
gecici = [m for m in tck_temiz if 'eçici' in m['madde_no']]
print(f"TCK: {len(tck_temiz)} articles ({len(gecici)} temporary)")

## TMK — Türk Medeni Kanunu (4721)

**Key challenges:**
- Mülga annotations mid-text (not at start) requiring inline removal
- Long articles (some >1024 tokens) — retained intact
- Trailing next-section titles extracted via last lowercase-ending sentence boundary

In [ ]:
def tmk_temizle(maddeler):
    """
    Clean TMK articles.
    - Removes inline mülga annotations (not just fully-mülga articles)
    - Extracts trailing section title using last sentence boundary
      (last match of lowercase-letter + period + space)
    """
    goruldu = set()
    sonuclar = []
    bekleyen_baslik = ""

    for madde in maddeler:
        metin = madde['text']
        madde_no = madde['madde_numarasi'].strip()

        if madde_no in goruldu:
            continue
        goruldu.add(madde_no)

        # Skip fully-mülga, rescue title
        metin_temiz = metin.strip().lstrip('–').lstrip('-').strip()
        if metin_temiz.lower().startswith('(mülga'):
            son_parantez = metin.rfind(')')
            if son_parantez != -1:
                baslik = metin[son_parantez + 1:].strip()
                if baslik:
                    bekleyen_baslik = baslik
            continue

        # Remove inline mülga annotations and footnotes
        metin = re.sub(r'\(Mülga[^)]*\)', '', metin, flags=re.IGNORECASE).strip()
        metin = re.sub(r'\[\d+\]', '', metin)
        metin = re.sub(r'\.\s+\d+\s+', '. ', metin)

        # Extract trailing title via last sentence boundary
        # (last occurrence of lowercase + period + space)
        eslesme = list(re.finditer(r'[a-züçğışöü]\.\s', metin))
        if eslesme:
            son = eslesme[-1]
            icerik = metin[:son.start() + 2].strip()
            sonraki_baslik = metin[son.end():].strip()
        else:
            icerik = metin.strip()
            sonraki_baslik = ""

        if bekleyen_baslik and sonuclar:
            sonuclar[-1]['sonraki_baslik'] = bekleyen_baslik
            sonuclar[-1]['encode_text'] = (
                sonuclar[-1]['icerik'] + f"\n\n{bekleyen_baslik}"
            )
            bekleyen_baslik = ""

        encode_text = icerik + (f"\n\n{sonraki_baslik}" if sonraki_baslik else "")

        sonuclar.append({
            "kanun": "Türk Medeni Kanunu",
            "madde_no": madde_no,
            "icerik": icerik,
            "sonraki_baslik": sonraki_baslik,
            "encode_text": encode_text
        })

    return sonuclar

tmk_temiz = tmk_temizle(ham_kanunlar["4721"]["maddeler"])
gecici = [m for m in tmk_temiz if 'eçici' in m['madde_no']]
print(f"TMK: {len(tmk_temiz)} articles ({len(gecici)} temporary)")

## İş Kanunu (4857)

**Key challenges:**
- Boilerplate transition tables ('4857 sayılı kanuna göre...') that must be excluded
- Mülga fıkralar with trailing section titles requiring rescue
- Temporary (Geçici) articles that must be preserved

In [ ]:
def isk_temizle(maddeler):
    """
    Clean İş Kanunu articles.
    - Filters boilerplate entries (transition tables, effective-date lists)
    - Handles both numbered-paragraph and non-numbered article formats
    - Rescues section titles from mülga fıkralar
    """
    goruldu = set()
    sonuclar = []
    bekleyen_baslik = ""

    # Boilerplate phrases to exclude entirely
    BOILERPLATE = [
        '4857 sayılı kanuna',
        'yürürlüğe giriş tarihlerini gösterir'
    ]

    for madde in maddeler:
        metin = madde['text']
        madde_no = madde['madde_numarasi'].strip()

        # Skip empty entries
        if len(metin.strip()) < 10:
            continue

        # Skip boilerplate
        if any(b in metin.lower() for b in BOILERPLATE):
            continue

        if madde_no in goruldu:
            continue
        goruldu.add(madde_no)

        # Skip fully-mülga, rescue title
        metin_temiz = metin.strip().lstrip('–').lstrip('-').strip()
        if metin_temiz.lower().startswith('(mülga'):
            son_parantez = metin.rfind(')')
            if son_parantez != -1:
                baslik = metin[son_parantez + 1:].strip()
                if baslik:
                    bekleyen_baslik = baslik
            continue

        # Remove footnotes
        metin = re.sub(r'\[\d+\]', '', metin)
        metin = re.sub(r'\.\s+\d+\s+', '. ', metin)

        # Handle numbered paragraphs vs. plain text
        if '(1)' in metin:
            fikralar = re.split(r'(?=\(\d+\))', metin)
            fikralar = [f.strip() for f in fikralar if f.strip()]
            temiz = []
            for f in fikralar:
                if 'mülga' in f.lower():
                    eslesme = re.search(r'\)\s+([A-ZÇĞİÖŞÜ][^()]+)$', f)
                    if eslesme:
                        bekleyen_baslik = eslesme.group(1).strip()
                else:
                    temiz.append(f)
            metin = ' '.join(temiz)
        else:
            metin = re.sub(r'\(Mülga[^)]*\)', '', metin, flags=re.IGNORECASE).strip()

        # Split trailing section title
        son_nokta = metin.rfind('.')
        if son_nokta == -1:
            icerik = metin.strip()
            sonraki_baslik = ""
        else:
            icerik = metin[:son_nokta + 1].strip()
            baslik_ham = metin[son_nokta + 1:].strip()
            sonraki_baslik = son_baslik_isle(baslik_ham)

        if bekleyen_baslik and sonuclar:
            sonuclar[-1]['sonraki_baslik'] = bekleyen_baslik
            sonuclar[-1]['encode_text'] = (
                sonuclar[-1]['icerik'] + f"\n\n{bekleyen_baslik}"
            )
            bekleyen_baslik = ""

        encode_text = icerik + (f"\n\n{sonraki_baslik}" if sonraki_baslik else "")

        sonuclar.append({
            "kanun": "İş Kanunu",
            "madde_no": madde_no,
            "icerik": icerik,
            "sonraki_baslik": sonraki_baslik,
            "encode_text": encode_text
        })

    return sonuclar

isk_temiz = isk_temizle(ham_kanunlar["4857"]["maddeler"])
gecici = [m for m in isk_temiz if 'eçici' in m['madde_no']]
print(f"İş K: {len(isk_temiz)} articles ({len(gecici)} temporary)")

## TBK — Türk Borçlar Kanunu (6098)

**Key challenges:**
- Sub-item markers (a., b., c.) can confuse sentence-boundary detection
- Inline mülga annotations requiring removal
- Trailing section title extraction via sentence boundary

In [ ]:
def tbk_temizle(maddeler):
    """
    Clean TBK articles.
    - Removes footnotes and inline mülga annotations
    - Extracts trailing section title via last sentence boundary
    """
    goruldu = set()
    sonuclar = []
    bekleyen_baslik = ""

    for madde in maddeler:
        metin = madde['text']
        madde_no = madde['madde_numarasi'].strip()

        if madde_no in goruldu:
            continue
        goruldu.add(madde_no)

        # Skip fully-mülga, rescue title
        metin_temiz = metin.strip().lstrip('–').lstrip('-').strip()
        if metin_temiz.lower().startswith('(mülga'):
            son_parantez = metin.rfind(')')
            if son_parantez != -1:
                baslik = metin[son_parantez + 1:].strip()
                if baslik:
                    bekleyen_baslik = baslik
            continue

        # Clean footnotes and inline mülga
        metin = re.sub(r'\[\d+\]', '', metin)
        metin = re.sub(r'\.\s+\d+\s+', '. ', metin)
        metin = re.sub(r'\(Mülga[^)]*\)', '', metin, flags=re.IGNORECASE).strip()

        # Extract trailing section title via last sentence boundary
        eslesme = list(re.finditer(r'[a-züçğışöü]\.\s', metin))
        if eslesme:
            son = eslesme[-1]
            icerik = metin[:son.start() + 2].strip()
            sonraki_baslik = metin[son.end():].strip()
        else:
            icerik = metin.strip()
            sonraki_baslik = ""

        if bekleyen_baslik and sonuclar:
            sonuclar[-1]['sonraki_baslik'] = bekleyen_baslik
            sonuclar[-1]['encode_text'] = (
                sonuclar[-1]['icerik'] + f"\n\n{bekleyen_baslik}"
            )
            bekleyen_baslik = ""

        encode_text = icerik + (f"\n\n{sonraki_baslik}" if sonraki_baslik else "")

        sonuclar.append({
            "kanun": "Türk Borçlar Kanunu",
            "madde_no": madde_no,
            "icerik": icerik,
            "sonraki_baslik": sonraki_baslik,
            "encode_text": encode_text
        })

    return sonuclar

tbk_temiz = tbk_temizle(ham_kanunlar["6098"]["maddeler"])
gecici = [m for m in tbk_temiz if 'eçici' in m['madde_no']]
print(f"TBK: {len(tbk_temiz)} articles ({len(gecici)} temporary)")

## Anayasa — Türkiye Cumhuriyeti Anayasası (2709)

**Key challenges:**
- Referendum boilerplate ('halk oylamasına sunulması') requiring exclusion
- 'İşlenemeyen hükümler' and 'yürürlüğe giriş' boilerplate tables
- Değişik (amended) annotations that should be stripped
- Geçici (temporary) articles that must be preserved

In [ ]:
def anayasa_temizle(maddeler):
    """
    Clean Anayasa articles.
    - Filters boilerplate entries (referendum notices, transition tables,
      işlenemeyen hükümler blocks)
    - Removes both Mülga and Değişik annotations
    - Preserves Geçici (temporary) articles
    """
    goruldu = set()
    sonuclar = []
    bekleyen_baslik = ""

    BOILERPLATE = [
        'bu kanun yayımı tarihinde yürürlüğe girer',
        'sayılı kanunun hükmüdür',
        'halk oylamasına sunulması',
        'işlenemeyen hükümler',
        'yürürlüğe giriş tarihlerini gösterir'
    ]

    for madde in maddeler:
        metin = madde['text']
        madde_no = madde['madde_numarasi'].strip()  # preserve original case

        # Skip empty entries
        if len(metin.strip()) < 10:
            continue

        # Skip boilerplate
        if any(b in metin.lower() for b in BOILERPLATE):
            continue

        # Deduplicate by raw madde_numarasi
        if madde_no in goruldu:
            continue
        goruldu.add(madde_no)

        # Skip fully-mülga, rescue title
        metin_temiz = metin.strip().lstrip('–').lstrip('-').strip()
        if metin_temiz.lower().startswith('(mülga'):
            son_parantez = metin.rfind(')')
            if son_parantez != -1:
                baslik = metin[son_parantez + 1:].strip()
                if baslik:
                    bekleyen_baslik = baslik
            continue

        # Remove footnotes, mülga and değişik annotations, normalize whitespace
        metin = re.sub(r'\[\d+\]', '', metin)
        metin = re.sub(r'\.\s+\d+\s+', '. ', metin)
        metin = re.sub(r'\(Mülga[^)]*\)', '', metin)
        metin = re.sub(r'\(Değişik[^)]*\)', '', metin)
        metin = re.sub(r'\s+', ' ', metin).strip()

        # Extract trailing section title via last sentence boundary
        eslesme = list(re.finditer(r'[a-züçğışöü]\.\s', metin))
        if eslesme:
            son = eslesme[-1]
            icerik = metin[:son.start() + 2].strip()
            sonraki_baslik = metin[son.end():].strip()
        else:
            icerik = metin.strip()
            sonraki_baslik = ""

        if bekleyen_baslik and sonuclar:
            sonuclar[-1]['sonraki_baslik'] = bekleyen_baslik
            sonuclar[-1]['encode_text'] = (
                sonuclar[-1]['icerik'] + f"\n\n{bekleyen_baslik}"
            )
            bekleyen_baslik = ""

        encode_text = icerik + (f"\n\n{sonraki_baslik}" if sonraki_baslik else "")

        sonuclar.append({
            "kanun": "Türkiye Cumhuriyeti Anayasası",
            "madde_no": madde_no,
            "icerik": icerik,
            "sonraki_baslik": sonraki_baslik,
            "encode_text": encode_text
        })

    return sonuclar

anayasa_temiz = anayasa_temizle(ham_kanunlar["2709"]["maddeler"])
gecici = [m for m in anayasa_temiz if 'eçici' in m['madde_no']]
print(f"Anayasa: {len(anayasa_temiz)} articles ({len(gecici)} temporary)")

## Merge & Save

In [ ]:
import json
from collections import Counter

# Merge all statutes
tumü = tck_temiz + tmk_temiz + isk_temiz + tbk_temiz + anayasa_temiz

print("Per-statute breakdown:")
for kanun, sayi in Counter(m['kanun'] for m in tumü).items():
    gecici = sum(1 for m in tumü if m['kanun'] == kanun and 'eçici' in m['madde_no'])
    print(f"  {kanun}: {sayi} articles ({gecici} temporary)")

print(f"\nTotal: {len(tumü)} chunks")

# Save
with open('mevzuat_temiz.json', 'w', encoding='utf-8') as f:
    json.dump(tumü, f, ensure_ascii=False, indent=2)

print("✅ mevzuat_temiz.json saved")

## Token Length Analysis

In [ ]:
from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("newmindai/Mursit-Large-TR-Retrieval")

token_sayilari = []
for chunk in tumü:
    tokens = tokenizer(chunk['encode_text'], return_tensors='pt', truncation=False)
    token_sayilari.append(tokens['input_ids'].shape[1])

print(f"Token statistics:")
print(f"  Mean:   {np.mean(token_sayilari):.1f}")
print(f"  Median: {np.median(token_sayilari):.1f}")
print(f"  Max:    {max(token_sayilari)}")
print(f"  >512:   {sum(1 for t in token_sayilari if t > 512)} articles")
print(f"  >1024:  {sum(1 for t in token_sayilari if t > 1024)} articles")

# Show articles exceeding 1024 tokens
print("\nArticles exceeding 1024 tokens (retained intact):")
for chunk, tokens in zip(tumü, token_sayilari):
    if tokens > 1024:
        print(f"  {chunk['kanun']} — {chunk['madde_no']}: {tokens} tokens")